In [1]:
import networkx as nx
import numpy as np
import pandas as pd 
import scipy.stats
import random


from tqdm import tqdm

#for copy
import copy

In [2]:
def perform_contagion_transfers_unweighted_version(graph, infected_list, mode='simple', threshold_complex=2, beta_simple=0.5, prob_complex=-1):
    """
    Simulates contagion transfers within a network for both simple and complex contagions.

    Parameters:
    - graph (networkx.Graph): The network graph over which the contagion spreads.
    - infected_list (list): List of nodes that are currently infected.
    - mode (str, optional): Specifies the mode of contagion ('simple' or 'complex'). Defaults to 'simple'.
    - threshold_complex (int, optional): The threshold number of infected neighbors required for a node to become infected in complex mode. Defaults to 2.
    - beta_simple (float, optional): The transmission probability for each infected neighbor in simple mode. Defaults to 0.5.
    - prob_complex (float, optional): The probability of infection in complex mode if the number of infected neighbors is below the threshold but greater than 1. Defaults to -1.

    Returns:
    (bool, list): A tuple containing a boolean indicating if there are new infections and the updated list of infected nodes.

    Note:
    - In 'simple' mode, the probability of a node being infected is influenced by the number of its infected neighbors and the transmission probability.
    - In 'complex' mode, a node is infected if the number of its infected neighbors meets or exceeds the threshold or, with a certain probability, if the number of infected neighbors is below the threshold but greater than one.
    """
    new_infection_marker = False
    new_infected_nodes = []
    all_nodes = list(graph.nodes())

    for node in all_nodes:
        if node in infected_list:
            continue

        infected_neighbors_count = sum(neighbor in infected_list for neighbor in graph.neighbors(node))

        if mode == 'simple':
            infection_probability = 1 - (1 - beta_simple)**infected_neighbors_count
            if random.uniform(0,1) <= infection_probability:
                new_infection_marker = True
                new_infected_nodes.append(node)

        elif mode == 'complex':
            if infected_neighbors_count >= threshold_complex:
                new_infection_marker = True
                new_infected_nodes.append(node)
            elif 1 < infected_neighbors_count and random.uniform(0,1) <= prob_complex:
                new_infection_marker = True
                new_infected_nodes.append(node)

    infected_list.extend(new_infected_nodes)
    return new_infection_marker, infected_list

In [3]:
def perform_complex_contagion_unweighted(threshold, graph, initially_infected, prob_complex):
    """
    Performs a complex contagion process on a graph and computes the corresponding
    extended persistent homology (EPH) based on the infection steps.

    Parameters:
    - threshold (int): The threshold number of infected neighbors required for a node to become infected.
    - graph (networkx.Graph): The network graph over which the contagion spreads.
    - initially_infected (list): List of nodes that are initially infected.
    - prob_complex (float): The probability of infection in complex mode if the number of infected neighbors is below the threshold but greater than 1.

    Returns:
    - list: A list of the mean lifetime of features in the computed extended persistent homology for each contagion step.

    Note:
    - The function iterates over the contagion process until all nodes are infected or no new infections occur.
    - It computes the EPH based on the order of infection of nodes, using the subgraph of infected nodes at each step.
    """
    # Initialize
    infected_list = initially_infected
    time_current = 0
    new_infections = True
    step_of_infection_per_node = [None] * len(graph.nodes)
    complex_list_EPH = []

    # Mark initially infected nodes with their infection step (time 0)
    for node_index in initially_infected:
        node_pos = list(graph.nodes).index(node_index)
        step_of_infection_per_node[node_pos] = time_current

    # Perform contagion until all nodes are infected or no new infections
    while new_infections and len(infected_list) < len(graph.nodes):
        time_current += 1
        new_infections, infected_list = perform_contagion_transfers_unweighted_version(
            graph, infected_list, mode='complex', threshold_complex=threshold, prob_complex=prob_complex)

        # Update infection steps for newly infected nodes
        for node_index in infected_list:
            node_pos = list(graph.nodes).index(node_index)
            if step_of_infection_per_node[node_pos] is None:
                step_of_infection_per_node[node_pos] = time_current
    
    return np.array(step_of_infection_per_node)

In [4]:

def perform_simple_contagion_unweighted(beta_simple, graph, initially_infected):
    """
    Performs a simple contagion process on a graph and computes the corresponding
    extended persistent homology (EPH) based on the infection steps.

    Parameters:
    - beta_simple (float): The transmission probability for each infected neighbor in simple mode.
    - graph (networkx.Graph): The network graph over which the contagion spreads.
    - initially_infected (list): List of nodes that are initially infected.

    Returns:
    - list: A list of the mean lifetime of features in the computed extended persistent homology for each contagion step.
    - mean of the above List
    - rho (correlation of PRL article)

    Note:
    - The function iterates over the contagion process until all nodes are infected or no new infections occur.
    - It computes the EPH based on the order of infection of nodes, using the subgraph of infected nodes at each step.
    """
    # Initialize
    infected_list = initially_infected[:]
    time_current = 0
    new_infections = True
    step_of_infection_per_node = [None] * len(graph.nodes)
    simple_list_EPH = []

    # Mark initially infected nodes with their infection step (time 0)
    for node_index in initially_infected:
        node_pos = list(graph.nodes).index(node_index)
        step_of_infection_per_node[node_pos] = time_current

    # Perform contagion until all nodes are infected or no new infections
    while new_infections and len(infected_list) < len(graph.nodes):
        time_current += 1
        new_infections, infected_list = perform_contagion_transfers_unweighted_version(
            graph, infected_list, mode='simple', beta_simple=beta_simple)

        # Update infection steps for newly infected nodes
        for node_index in infected_list:
            node_pos = list(graph.nodes).index(node_index)
            if step_of_infection_per_node[node_pos] is None:
                step_of_infection_per_node[node_pos] = time_current

    return np.array(step_of_infection_per_node)

## Complex contagion 

In [5]:
threshold_set = [2,3,4,5,6]
q_set = [-1,0.1,0.2,0.3,0.4,0.5]

In [6]:
network_list = [
    'conf','email_eu','hospital','school','work'
]

In [7]:
def init_nodes_var_reduction(iter_cnt,arr,k):
    rng = np.random.RandomState(iter_cnt)
    init_nodes = rng.choice(arr,k)
    return list(init_nodes)

In [8]:
for network_name in network_list:
    
    #reading graph 
    G = nx.read_graphml(f'../networks/unweighted/G_unweighted_{network_name}.graphml')
    
    #preprocessing
    # Create a mapping from current node names (str) to integers
    mapping = {node: int(float(node)) for node in G.nodes()}
    # Relabel the nodes in the graph using the mapping
    G = nx.relabel_nodes(G, mapping)

         
    #storing simulations
    order_collection_simulations = [] 
    seed_sim = [] 
    theta_sim = [] 
    q_sim = [] 
    
    #doing simulations
    for q_curr in tqdm(q_set):
        for sample_it in range(100):
            #initilize graph and infected nodes
            list_inf = init_nodes_var_reduction(iter_cnt=sample_it,
                                                arr=list(G.nodes),k=10)
            th = init_nodes_var_reduction(iter_cnt=sample_it,arr=threshold_set,k=1)[0]
#             q_curr = -1
            
            order = perform_complex_contagion_unweighted(threshold=th, graph=copy.deepcopy(G), 
                                                       initially_infected=copy.deepcopy(list_inf), 
                                                       prob_complex=q_curr)

            order_collection_simulations.append(order)
            seed_sim.append(sample_it)
            theta_sim.append(th)
            q_sim.append(q_curr)
            
            
    
    result = pd.DataFrame(order_collection_simulations)
    result['theta'] = theta_sim
    result['seed'] = seed_sim
    result['q'] = q_sim
    
    #saving 
    result.to_csv(f'../results/unweighted/given_q/unweighted_complex_{network_name}.csv')    

100%|██████████| 6/6 [00:06<00:00,  1.07s/it]


## Simple contagion 

In [9]:
betha_set = [0.4,0.5,0.6]

In [10]:
network_list = [
    'conf','email_eu','hospital','school','work'
]

In [11]:
def init_nodes_var_reduction(iter_cnt,arr,k):
    rng = np.random.RandomState(iter_cnt)
    init_nodes = rng.choice(arr,k)
    return list(init_nodes)

In [12]:
for network_name in tqdm(network_list):
    
    #reading graph 
    G = nx.read_graphml(f'../networks/unweighted/G_unweighted_{network_name}.graphml')
    
    #preprocessing
    # Create a mapping from current node names (str) to integers
    mapping = {node: int(float(node)) for node in G.nodes()}
    # Relabel the nodes in the graph using the mapping
    G = nx.relabel_nodes(G, mapping)
        
            
    #storing simulations
    order_collection_simulations = [] 
    seed_sim = [] 
    betha_sim = [] 
    
    #doing simulations
    for sample_it in range(100):
        #initilize graph and infected nodes
        beta = init_nodes_var_reduction(iter_cnt=sample_it,arr=betha_set,k=1)[0]

        list_inf = init_nodes_var_reduction(iter_cnt=sample_it,
                                            arr=list(G.nodes),k=10)
        order = perform_simple_contagion_unweighted(beta_simple=beta, graph=copy.deepcopy(G), 
                                                        initially_infected=copy.deepcopy(list_inf))

        order_collection_simulations.append(order)
        seed_sim.append(sample_it)
        betha_sim.append(beta)


    
    result = pd.DataFrame(order_collection_simulations)
    result['betha'] = betha_sim
    result['seed'] = seed_sim

    #saving 
    result.to_csv(f'../results/unweighted/given_q/unweighted_simple_{network_name}.csv')


100%|██████████| 5/5 [01:39<00:00, 19.97s/it]


In [13]:
2

2